In [1]:
from pathlib import Path
import sys
import yaml

from data_prep.download import download_from_config, verify_download_path

# Always resolve the workspace root and keep downloads under the repo's staging area.
workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

raw_dir = (workspace_root / "data" / "raw").resolve()
raw_dir.mkdir(parents=True, exist_ok=True)

dataset_name = "olist"
dataset_dir = (raw_dir / dataset_name).resolve()
dataset_dir.mkdir(parents=True, exist_ok=True)

config_dir = (workspace_root / "configs").resolve()
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "olist.yaml"

config = {
    "version": 1,
    "dataset_name": dataset_name,
    "operation": "download",
    "source": {
        "type": "kaggle",
        "kaggle_dataset": "olistbr/brazilian-ecommerce",
    },
    "destination": {
        "path": str(dataset_dir),
        "format": "csv",
    },
}
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print("Using Python:", sys.executable)
print("Workspace root:", workspace_root)
print("Download directory:", dataset_dir)
print("Configured config:", config_path)

try:
    result = download_from_config(config_path)
    verify_download_path(result)
except RuntimeError as exc:
    raise RuntimeError(
        "The Olist download failed because the notebook is not running under the project environment or Kaggle credentials are missing. "
        "Use the repo .venv and ensure ~/.kaggle/kaggle.json or ~/.kaggle/access_token exists before retrying."
    ) from exc

print("Download result:", result)

Using Python: /home/rajiv/programming/kmds-dataset-util/.venv/bin/python
Workspace root: /home/rajiv/programming/kmds-dataset-util
Download directory: /home/rajiv/programming/kmds-dataset-util/data/raw/olist
Configured config: /home/rajiv/programming/kmds-dataset-util/configs/olist.yaml
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


100%|██████████| 42.6M/42.6M [00:03<00:00, 11.9MB/s]



Download result: /home/rajiv/programming/kmds-dataset-util/data/raw/olist


In [2]:
from pathlib import Path
import pandas as pd

base = Path.cwd().resolve()
if base.name == "notebooks":
    base = base.parent

olist_dir = (base / "data" / "raw" / "olist").resolve()
olist_dir.mkdir(parents=True, exist_ok=True)

rows = []
for csv_path in sorted(olist_dir.glob("*.csv")):
    if csv_path.name == "data_dictionary.csv":
        continue
    try:
        sample = pd.read_csv(csv_path, nrows=5)
    except Exception:
        continue
    for col in sample.columns:
        rows.append(
            {
                "attribute": col,
                "description": f"{csv_path.stem} field: {col}",
                "data_type": str(sample[col].dtype),
                "source_file": csv_path.name,
            }
        )

seen = {}
for row in rows:
    key = row["attribute"]
    if key not in seen:
        seen[key] = row

dictionary_df = pd.DataFrame(list(seen.values()))
dictionary_df = dictionary_df[["attribute", "description", "data_type", "source_file"]]
out_path = olist_dir / "data_dictionary.csv"
dictionary_df.to_csv(out_path, index=False)

print(f"Generated aggregated dictionary at: {out_path}")
print(dictionary_df.head(10).to_string(index=False))

Generated aggregated dictionary at: /home/rajiv/programming/kmds-dataset-util/data/raw/olist/data_dictionary.csv
                  attribute                                                  description data_type                   source_file
                  attribute                         data_dictionary_raw field: attribute       str       data_dictionary_raw.csv
                description                       data_dictionary_raw field: description       str       data_dictionary_raw.csv
                  data_type                         data_dictionary_raw field: data_type       str       data_dictionary_raw.csv
                customer_id                   olist_customers_dataset field: customer_id       str   olist_customers_dataset.csv
         customer_unique_id            olist_customers_dataset field: customer_unique_id       str   olist_customers_dataset.csv
   customer_zip_code_prefix      olist_customers_dataset field: customer_zip_code_prefix     int64   olist_custom

This data dictionary was generated in the raw Olist staging directory using the dataset files available in the Kaggle download and the prompt provided to the Copilot coding assistant.